In [58]:
import importlib
import torch
import os
import pandas as pd
import src.dataset.dataset as dataset_module
import src.models.model as model_module
importlib.reload(dataset_module)
importlib.reload(model_module)
from src.dataset.dataset import GalaxyDataset
from torch.utils.data import DataLoader, dataset
from torchvision import transforms
from src.models.model import GalaxyClassifier_ResNet
import random

In [20]:
raw_transforms = transforms.Compose([
    transforms.CenterCrop(random.randint(170, 240)),  
    transforms.Resize(160),                            
    transforms.RandomCrop(128),                        
    transforms.RandomRotation(degrees = [90, 180]),
    transforms.RandomRotation(degrees = [0,270]),      
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.1376, 0.1276, 0.1187],
                         std= [0.1728, 0.1586, 0.1488])    
])



In [64]:
labels = labels_df = pd.read_csv(r"D:\galaxy classification\data\preprocessed\labels\labeled_data_1.csv")
dataset = GalaxyDataset(img_dir= r"D:\galaxy classification\images", labels_df=labels_df, transform=raw_transforms)

In [34]:
sampler, class_weights = dataset.make_weighted_sampler()
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4, sampler=sampler)

In [65]:
sample_size = min(100, len(dataset))
if sample_size == 0:
    raise ValueError("No valid images were found in data/preprocessed/images. Run 'dvc pull' or point GalaxyDataset at a populated image directory.")

indices = random.sample(range(len(dataset)), sample_size)

subset_dataset = torch.utils.data.Subset(dataset, indices)
subset_sampler, _ = dataset.make_weighted_sampler(subset_dataset.indices)

In [66]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GalaxyClassifier_ResNet(num_classes=5).to(device)


optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer,T_mult = 2, eta_min=0.0001 ,T_0=10)

_,class_weights = dataset.make_weighted_sampler()

criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1,weight = class_weights.to(device))


In [67]:
subset_dataloader = DataLoader(subset_dataset, batch_size=32, shuffle=False,sampler = subset_sampler, num_workers=4)

In [71]:
image,size = subset_dataset[0]

In [72]:
image.shape

torch.Size([3, 128, 128])

In [68]:
for epoch in range(10):
    model.train()
    for images, labels in subset_dataloader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        print(f"Epoch [{epoch+1}/10], Loss: {loss.item():.4f}")
    
    scheduler.step()

RuntimeError: Given groups=1, weight of size [1024, 512, 1, 1], expected input[32, 64, 32, 32] to have 512 channels, but got 64 channels instead